In [1]:
# Import Packages
import pandas as pd
import re

# Import from local files
from data_cleaning import DataCleaner
from extraction import extract_data
from entity_resolution import EntityResolver

resolver = EntityResolver()

# Set pandas display options
pd.set_option("display.width", 1000)
pd.set_option("display.max_columns", None)


# ----------------------------------------------------------
# Extract
# ----------------------------------------------------------

extracted_df = extract_data(
    "2025 Roll - Parcels Export.xlsx",
    "Only Delinquent Parcels"
)


# ----------------------------------------------------------
# Clean
# ----------------------------------------------------------

cleaner = DataCleaner()

clean_df = cleaner.clean_data(
    extracted_df
)


# ----------------------------------------------------------
# Entity Resolution
# ----------------------------------------------------------

(
    entity_occurrences,
    entities,
    parcel_entity_roles,
    association_evidence,
    entity_associations
) = EntityResolver().build(
    clean_df
)

# Create audit dataframe.
resolution_audit = (
    entity_occurrences
    .merge(
        entities[
            [
                "entity_id",
                "canonical_name",
                "canonical_address",
                "entity_type",
                "occurrence_count"
            ]
        ],
        on="entity_id",
        how="left",
        validate="many_to_one",
        suffixes=("_source", "_final")
    )
)

resolution_audit.head()

,occurrence_id,parcel_number,entity_name,associated_address,role,source_entity_name,source_column,source_occurrence_id,entity_type_source,joint_parse_rule,joint_parse_confidence,joint_expanded,joint_party_index,joint_party_count,organization_type,organization_classification_confidence,address_source,entity_id,canonical_name,canonical_address,entity_type_final,occurrence_count
0,1,01-01-201-013,POLOMCAK CLAUDIA,"31029 30TH ST, PAW PAW, MI 49079",Owner,POLOMCAK CLAUDIA,owner_name,1,Person,,,False,NaN,NaN,NaN,NaN,owner_address,3628,POLOMCAK CLAUDIA,"31029 30TH ST, PAW PAW, MI 49079",Person,2
1,2,01-01-251-013,CLAWSON DORRINDA &,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA &,owner_name,2,Joint,,,False,NaN,NaN,NaN,NaN,owner_address,1018,CLAWSON DORRINDA &,"10591 N 12TH ST, PLAINWELL, MI 49080",Joint,2
2,3,01-01-251-050,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA L,owner_name,3,Person,,,False,NaN,NaN,NaN,NaN,owner_address,1019,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Person,6
3,4,01-01-251-060,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Owner,CLAWSON DORRINDA L,owner_name,4,Person,,,False,NaN,NaN,NaN,NaN,owner_address,1019,CLAWSON DORRINDA L,"10591 N 12TH ST, PLAINWELL, MI 49080",Person,6
4,5,01-01-301-050,"SPIEKERMANN, TIM","10238 N 10TH ST, PLAINWELL, MI 49080",Owner,SPIEKERMANN TIM & JUDY,owner_name,5,Person,shared_surname_single_right_name,High,True,1.0,2.0,NaN,NaN,owner_address,4303,"SPIEKERMANN, TIM","10238 N 10TH ST, PLAINWELL, MI 49080",Person,2


## Entities with the highest occurrence_count

In [2]:
entities.sort_values(
    "occurrence_count",
    ascending=False
).head(10)

,entity_id,canonical_name,entity_type,canonical_address,organization_type,organization_classification_confidence,occurrence_count
2379,2380,KAYLOR HOLDINGS LLC,Organization,"1405 HOLIDAY LN, PORTAGE, MI 49024",LLC,High,62
667,668,"BROOKS, SAMUEL C",Person,"804 CHARLOTTE AVE, KALAMAZOO, MI 49048",NaN,NaN,51
4335,4336,STATE WIDE RENTALS LLC,Organization,"606 BRYANT ST, KALAMAZOO, MI 49001",LLC,High,25
20,21,"19 PROPS, LLC",Organization,"200 W MICHIGAN AVE STE 201, KALAMAZOO, MI 49007",LLC,High,21
1718,1719,"GIBSON, ROBERT M",Person,"8713 S 38TH ST, SCOTTS, MI 49088",NaN,NaN,21
1715,1716,"GIBSON, MARTHA M",Person,"8713 S 38TH ST, SCOTTS, MI 49088",NaN,NaN,19
484,485,"BISHOP, BRADLEY J",Person,"PO BOX 312, PORTAGE, MI 49081",NaN,NaN,19
2018,2019,"HERTEL, JOSEPH B",Person,"2326 S ROSE ST, KALAMAZOO, MI 49001",NaN,NaN,18
4805,4806,WALLY BROWN LLC,Organization,"563 TREASURE ISLAND DR, MATTAWAN, MI 49071",LLC,High,18
5111,5112,"YOUNG, CAVEL J",Person,"2111 N DRAKE RD, KALAMAZOO, MI 49006",NaN,NaN,18


## Entities with canonical names containing a middle name or middle initial

In [3]:
SUFFIXES = {
    "JR",
    "SR",
    "JUNIOR",
    "SENIOR",
    "II",
    "III",
    "IV",
}


def get_middle_tokens(name):
    """
    Return possible middle-name tokens from a person name.

    Supports both:
        SMITH, JOHN A
        SMITH JOHN A

    This is for manual inspection only, not entity-resolution logic.
    """

    if pd.isna(name):
        return tuple()

    name = str(name).upper().strip()
    name = name.replace(".", "")

    # Skip obvious joint/compound records
    if (
        "&" in name
        or "/" in name
        or re.search(r"\bAND\b", name)
    ):
        return tuple()

    if "," in name:

        _, given_part = name.split(",", 1)
        tokens = given_part.strip().split()

        if tokens and tokens[-1] in SUFFIXES:
            tokens = tokens[:-1]

        # FIRST [MIDDLE...]
        return tuple(tokens[1:])

    else:

        tokens = name.split()

        if tokens and tokens[-1] in SUFFIXES:
            tokens = tokens[:-1]

        # Assume LAST FIRST [MIDDLE...]
        if len(tokens) < 3:
            return tuple()

        return tuple(tokens[2:])

people_with_middle_names = (
    entities[
        entities["entity_type"] == "Person"
    ]
    .copy()
)

people_with_middle_names["middle_tokens"] = (
    people_with_middle_names[
        "canonical_name"
    ]
    .apply(get_middle_tokens)
)

people_with_middle_names = (
    people_with_middle_names[
        people_with_middle_names[
            "middle_tokens"
        ].apply(len) > 0
    ]
    .sort_values(
        "occurrence_count",
        ascending=False
    )
)

people_with_middle_names[
    [
        "entity_id",
        "canonical_name",
        "middle_tokens",
        "canonical_address",
        "occurrence_count"
    ]
]

,entity_id,canonical_name,middle_tokens,canonical_address,occurrence_count
667,668,"BROOKS, SAMUEL C","(C,)","804 CHARLOTTE AVE, KALAMAZOO, MI 49048",51
1718,1719,"GIBSON, ROBERT M","(M,)","8713 S 38TH ST, SCOTTS, MI 49088",21
484,485,"BISHOP, BRADLEY J","(J,)","PO BOX 312, PORTAGE, MI 49081",19
1715,1716,"GIBSON, MARTHA M","(M,)","8713 S 38TH ST, SCOTTS, MI 49088",19
2018,2019,"HERTEL, JOSEPH B","(B,)","2326 S ROSE ST, KALAMAZOO, MI 49001",18
...,...,...,...,...,...
3818,3819,"RICHARDS, JEAN M","(M,)","7383 W OLD CHANNEL TRL, MONTAGUE, MI 49437",1
3817,3818,"RICHARDS, DONALD L","(L,)","89513 SHORELANE DR, LAWTON, MI 49065",1
3816,3817,"RICHARDS, DONALD L","(L,)","7383 W OLD CHANNEL TRL, MONTAGUE, MI 49437",1
3792,3793,"REILLY, VALERIE LYNN","(LYNN,)","817 HOFFMAN PL, KALAMAZOO, MI 49007",1


- Some names have commas while others don't
    - For example, KAVANAUGH, JAMES J vs. CLAWSON DORRINDA L
        - Note that both had their middle initial listed correctly under middle_tokens
- The canonical name FRITH, TERRANCE JEROME A produced the middle_token (JEROME, A)
- The canonical name BROWN, ROBERT WENDELL, JR. produced the middle_token (WENDELL,,)
    - Note that there are two commas rather than the usual one
- GREATER TRUE WORSHIP COGIC produced (WORSHIP, COGIC)
- BLACKSTAR STABILITY REVITALIZATION produced (REVITALIZATION,)
- EDWARDS, KARRIE; EDWARDS, KENTON produced (EDWARDS,, KENTON)

Unrelated case:
There are two entities with the name WIDGER JAMES LEROY, one with the address 214 MARSHALL ST, ALLEGAN, MI 49010 and the other with the address 214 MARSHALL ST, ALLEGAN, MI 49010. Because they have the exact same name and their address, with the removal of ST from the first case, are the same - they should clearly be merged.

### Distinguishing middle initials from full middle names

In [4]:
def middle_type(tokens):

    if not tokens:
        return None

    if all(len(token) == 1 for token in tokens):
        return "Initial"

    return "Full/Other"


people_with_middle_names["middle_type"] = (
    people_with_middle_names[
        "middle_tokens"
    ]
    .apply(middle_type)
)

people_with_middle_names[
    "middle_type"
].value_counts()

middle_type
Initial       1421
Full/Other     411
Name: count, dtype: int64

Out of 3,955 unique entity names, 1,240 show as containing a middle name or initial.

In [5]:
797 + 443

1240

## Entities the previously had a blank address

In [6]:
blank_address_cases = (
    resolution_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ],
        dropna=False
    )
    .agg(
        had_blank_address=(
            "associated_address",
            lambda x: (
                x.fillna("")
                .str.strip()
                .eq("")
                .any()
            )
        ),
        source_addresses=(
            "associated_address",
            lambda x: tuple(
                sorted(
                    {
                        address
                        for address in x.fillna("")
                        if address != ""
                    }
                )
            )
        ),
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        occurrence_count=(
            "occurrence_id",
            "size"
        )
    )
    .reset_index()
)

blank_address_cases = (
    blank_address_cases[
        blank_address_cases["had_blank_address"]
        &
        (
            blank_address_cases[
                "canonical_address"
            ] != ""
        )
    ]
)

print(blank_address_cases.shape)
blank_address_cases.head(50)

(1923, 7)


,entity_id,canonical_name,canonical_address,had_blank_address,source_addresses,source_names,occurrence_count
15,16,1733 W. MAIN ST LLC,"1733 1/2 W MAIN ST, KALAMAZOO, MI 49006",True,"(1733 1/2 W MAIN ST, KALAMAZOO, MI 49006,)","(1733 W. MAIN ST LLC,)",2
19,20,19 PROPS LLC,"200 W MICHIGAN AVE STE 201, KALAMAZOO, MI 49007-",True,"(200 W MICHIGAN AVE STE 201, KALAMAZOO, MI 490...","(19 PROPS LLC,)",2
21,22,"1930 WEST MAIN, LLC","3260 NILES RD STE B, SAINT JOSEPH, MI 49085",True,"(3260 NILES RD STE B, SAINT JOSEPH, MI 49085,)","(1930 WEST MAIN, LLC,)",2
24,25,2219 HUMPHREY LLC,"100 W MICHIGAN AVE STE 200, KALAMAZOO, MI 49007",True,"(100 W MICHIGAN AVE STE 200, KALAMAZOO, MI 490...","(2219 HUMPHREY LLC,)",2
26,27,232 GRAND LLC,"11670 RAMSEYER DR, PLAINWELL, MI 49080",True,"(11670 RAMSEYER DR, PLAINWELL, MI 49080,)","(232 GRAND LLC,)",2
28,29,26TH ST PROPERTIES LLC,"2915 E GRAND RIVER AVE, HOWELL, MI 48843",True,"(2915 E GRAND RIVER AVE, HOWELL, MI 48843,)","(26TH ST PROPERTIES LLC,)",2
38,39,3030 S 9TH LLC,"PO BOX 312098, NEW BRAUNFELS, TX 78131",True,"(PO BOX 312098, NEW BRAUNFELS, TX 78131,)","(3030 S 9TH LLC,)",2
41,42,3381 FOREST VIEW INVESTORS LLC,"2265 LIVERNOIS STE 500, TROY, MI 48083",True,"(2265 LIVERNOIS STE 500, TROY, MI 48083,)","(3381 FOREST VIEW INVESTORS LLC,)",4
42,43,"3501 LAKE HOLDINGS, LLC","3501 LAKE ST, KALAMAZOO, MI 49048",True,"(3501 LAKE ST, KALAMAZOO, MI 49048,)","(3501 LAKE HOLDINGS, LLC,)",2
49,50,5752 TRUST,"4426 HUGH HOWELL RD # B-369, TUCKER, GA 30084",True,"(4426 HUGH HOWELL RD # B-369, TUCKER, GA 30084,)","(5752 TRUST,)",2


There are 1,580 cases of entities that used to have a blank address
- **QUESTION: Is the main thing to look for here that the source_names and canonical_name are in fact similar enough that we would want to merge them?**

## Names affected by CORP/CORPORATION, INC/INCORPORATED, CO/COMPANY

In [7]:
ENTITY_NAME_REPLACEMENTS = {
    "CORP": "CORPORATION",
    "INC": "INCORPORATED",
    "CO": "COMPANY"
}


def expand_entity_abbreviations(name):

    if pd.isna(name):
        return ""

    name = str(name).upper().strip()

    name = name.replace(".", "")

    name = re.sub(r"\s+", " ", name)
    name = re.sub(r"\s*,\s*", ", ", name)

    for old, new in ENTITY_NAME_REPLACEMENTS.items():

        name = re.sub(
            rf"\b{re.escape(old)}\b",
            new,
            name
        )

    return name.strip()

abbreviation_audit = resolution_audit.copy()

abbreviation_audit["expanded_name"] = (
    abbreviation_audit[
        "entity_name"
    ]
    .apply(expand_entity_abbreviations)
)

abbreviation_cases = (
    abbreviation_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ]
    )
    .agg(
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        expanded_names=(
            "expanded_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        source_name_count=(
            "entity_name",
            "nunique"
        ),
        expanded_name_count=(
            "expanded_name",
            "nunique"
        )
    )
    .reset_index()
)

abbreviation_cases = (
    abbreviation_cases[
        (abbreviation_cases["source_name_count"] > 1)
        &
        (abbreviation_cases["expanded_name_count"] == 1)
    ]
)

abbreviation_cases.head(50)

,entity_id,canonical_name,canonical_address,source_names,expanded_names,source_name_count,expanded_name_count
113,114,AIM CORPORATION,"218 GLADYS ST, PORTAGE, MI 49002","(AIM CORP, AIM CORPORATION)","(AIM CORPORATION,)",2,1
2198,2199,J. SAJE LLC,"2900 KENSINGTON DR, KALAMAZOO, MI 49008","(J SAJE LLC, J. SAJE LLC)","(J SAJE LLC,)",2,1
2783,2784,MAIN QSR INC.,"1946 45TH ST STE C, MUNSTER, IN 46321","(MAIN QSR INC, MAIN QSR INC.)","(MAIN QSR INCORPORATED,)",2,1
3382,3383,"ONE DAY, LLC.","1417 ACADEMY, KALAMAZOO, MI 49006","(ONE DAY, LLC, ONE DAY, LLC.)","(ONE DAY, LLC,)",2,1
3981,3982,"S.T.D. MANAGEMENT, LLC","3001 10TH ST, WAYLAND, MI 49348","(S.T.D. MANAGEMENT, LLC, STD MANAGEMENT, LLC)","(STD MANAGEMENT, LLC,)",2,1


- There is 1 case of a CORP/CORPORATION merger.
- Other than that, the other 5 cases involve merging source names that differ by the inclusion/exclusion of a period.
    - For example, ONE DAY, LLC and ONE DAY, LLC.

## Suffix cases

### All people containing a suffix

In [8]:
suffix_pattern = (
    r"\b(?:JR|SR|JUNIOR|SENIOR|II|III|IV)\b"
)

suffix_entities = (
    entities[
        (entities["entity_type"] == "Person")
        &
        (
            entities["canonical_name"]
            .str.contains(
                suffix_pattern,
                regex=True,
                na=False
            )
        )
    ]
    .sort_values(
        "occurrence_count",
        ascending=False
    )
)

print(suffix_entities.shape)

suffix_entities[
    [
        "entity_id",
        "canonical_name",
        "canonical_address",
        "occurrence_count"
    ]
].head(50)

(125, 7)


,entity_id,canonical_name,canonical_address,occurrence_count
176,177,"ANDERSON, A DAVID SR","PO BOX 50204, KALAMAZOO, MI 49005-0204",15
3676,3677,"PRIMER, DANIEL L JR","6841 N SPRINKLE RD, KALAMAZOO, MI 49004",12
2482,2483,"KNOX, DANNY LEE SR","6336 RAVINE RD, KALAMAZOO, MI 49009-9002",9
2218,2219,"JACKSON, SOL JR","5370 LUCERNE AVE, KALAMAZOO, MI 49048",8
1296,1297,"DERRING III, ALBERT","3041 N 26TH ST, KALAMAZOO, MI 49048",7
2676,2677,"LINDSEY, WILLIAM B JR","804 HARRISON ST, KALAMAZOO, MI 49007",6
694,695,"BROWN, ROBERT WENDELL, JR.","1115 DUPONT AVE, KALAMAZOO, MI 49048-3213",5
1219,1220,"DAVIS, KEITH SR.","3850 ROLLRIDGE, KALAMAZOO, MI 49004",5
4798,4799,WALL ROBERT JR,"1057 N 30TH ST, GALESBURG, MI 49053",4
546,547,"BOURNER, RALPH B JR","6243 TWILIGHT AVE, KALAMAZOO, MI 49048",4


- 146 entities contain suffixes

### Entities with both suffix and suffix-free versions

In [9]:
suffix_audit = (
    resolution_audit[
        resolution_audit[
            "entity_type_final"
        ] == "Person"
    ]
    .copy()
)

suffix_audit["has_suffix"] = (
    suffix_audit[
        "entity_name"
    ]
    .str.contains(
        suffix_pattern,
        regex=True,
        na=False
    )
)

suffix_merge_cases = (
    suffix_audit
    .groupby(
        [
            "entity_id",
            "canonical_name",
            "canonical_address"
        ]
    )
    .agg(
        source_names=(
            "entity_name",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        has_suffix_version=(
            "has_suffix",
            "any"
        ),
        has_no_suffix_version=(
            "has_suffix",
            lambda x: (~x).any()
        )
    )
    .reset_index()
)

suffix_merge_cases = (
    suffix_merge_cases[
        suffix_merge_cases["has_suffix_version"]
        &
        suffix_merge_cases["has_no_suffix_version"]
    ]
)

suffix_merge_cases.head(50)

,entity_id,canonical_name,canonical_address,source_names,has_suffix_version,has_no_suffix_version
1030,1297,"DERRING III, ALBERT","3041 N 26TH ST, KALAMAZOO, MI 49048","(DERRING ALBERT III, DERRING III, ALBERT, DERR...",True,True
2147,2677,"LINDSEY, WILLIAM B JR","804 HARRISON ST, KALAMAZOO, MI 49007","(LINDSEY, WILLIAM, LINDSEY, WILLIAM B, LINDSEY...",True,True


- There is 1 case of an entity merged due to having the same name and address, with different suffixes
    - Albert Derring and Albert Derring III

## Cases of finalized entities with the same name but different associated addresses

In [10]:
person_entities = (
    entities[
        entities["entity_type"] == "Person"
    ]
    .copy()
)

known_address_people = (
    person_entities[
        person_entities[
            "canonical_address"
        ].fillna("").str.strip() != ""
    ]
)

common_names_multiple_addresses = (
    known_address_people
    .groupby("canonical_name")
    .agg(
        entity_count=(
            "entity_id",
            "nunique"
        ),
        address_count=(
            "canonical_address",
            "nunique"
        ),
        addresses=(
            "canonical_address",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        entity_ids=(
            "entity_id",
            lambda x: tuple(
                sorted(set(x))
            )
        ),
        total_occurrences=(
            "occurrence_count",
            "sum"
        )
    )
    .reset_index()
)

common_names_multiple_addresses = (
    common_names_multiple_addresses[
        common_names_multiple_addresses[
            "address_count"
        ] > 1
    ]
    .sort_values(
        [
            "address_count",
            "total_occurrences"
        ],
        ascending=False
    )
)

print(common_names_multiple_addresses.shape)
common_names_multiple_addresses.head(50)

(351, 6)


,canonical_name,entity_count,address_count,addresses,entity_ids,total_occurrences
2449,"OWENS, LESSIE T",6,6,"(1214 N CHURCH ST, KALAMAZOO, MI 49007, 1214 N...","(3425, 3426, 3427, 3428, 3429, 3430)",9
2878,"SANTIAGO, JORGE",5,5,"(5101 SWALLOW AVE, KALAMAZOO, MI 49009, 5350 G...","(4027, 4028, 4029, 4030, 4031)",19
2880,"SANTIAGO, MARIA",4,4,"(5101 SWALLOW AVE, KALAMAZOO, MI 49009, 5350 G...","(4036, 4037, 4038, 4039)",18
183,"BALL, RONALD",4,4,"(1004 WOODWARD AVE, KALAMAZOO, MI 49007, 1821 ...","(320, 321, 322, 323)",5
2335,NEMIRE JAMES J,4,4,"(7167 SAGINAW DR, COMSTOCK, MI 49041, 7167 SAG...","(3256, 3257, 3258, 3259)",5
1186,"GATES, JEROME",3,3,"(1229 ORCHID AVE, SAINT LOUIS, MO 63147, 1229 ...","(1683, 1684, 1685)",18
2154,"MEIER, ARTHUR C",3,3,"(1226 CADET LN, KALAMAZOO, MI 49009, 1805 E MA...","(2994, 2995, 2996)",15
426,"BRINNEMAN, MICHAEL J",3,3,"(1802 FULFORD ST, KALAMAZOO, MI 49001, 5037 ST...","(639, 640, 641)",11
3422,"WADE, GEORGE R",3,3,"(1102 S BURDICK ST, KALAMAZOO, MI 49001, 1102 ...","(4775, 4776, 4777)",10
1731,"KERNELL, BRIANT",3,3,"(10238 PORTAGE RD, PORTAGE, MI 49002, 10238 PO...","(2421, 2422, 2423)",7


- There are 296 canonical_names which are associated with more than one entity/address

## Inspecting name formatting

In [11]:
people = (
    entity_occurrences[
        entity_occurrences["entity_type"] == "Person"
    ]
)

comma_names = (
    people["entity_name"]
    .str.contains(",", regex=False, na=False)
)

print(
    f"Person occurrences with comma-format names: "
    f"{comma_names.sum():,}"
)

print(
    f"Person occurrences without comma-format names: "
    f"{(~comma_names).sum():,}"
)

print(
    f"Percent parseable by current comma requirement: "
    f"{comma_names.mean():.1%}"
)

Person occurrences with comma-format names: 7,996
Person occurrences without comma-format names: 1,593
Percent parseable by current comma requirement: 83.4%


## Inspecting unknown entity types

In [12]:
entity_occurrences[
    "entity_type"
].value_counts()

entity_type
Person          9589
Organization    1978
Joint            469
Unknown           22
Name: count, dtype: int64

In [13]:
unknown_names = (
    entity_occurrences.loc[
        entity_occurrences["entity_type"] == "Unknown",
        "entity_name"
    ]
    .value_counts()
    .rename_axis("entity_name")
    .reset_index(name="occurrence_count")
)

print(
    f"Unique Unknown names: "
    f"{len(unknown_names):,}"
)

unknown_names.head(100)

Unique Unknown names: 12


,entity_name,occurrence_count
0,PAYMENT IN LIEU OF TAXES,4
1,"MCNEES, JR",3
2,"MI 4004 DOUGLAS AVE. KALAMAZOO, LL",2
3,CENTURYLINK,2
4,TRSTEE,2
5,RYAN,2
6,AIM,2
7,BENCHMARK,1
8,COLLABORATIVE,1
9,PO BOX 183,1


In [14]:
joint_names = (
    entity_occurrences.loc[
        entity_occurrences[
            "entity_type"
        ] == "Joint",
        "entity_name"
    ]
    .drop_duplicates()
)

joint_names.sample(
    min(100, len(joint_names)),
    random_state=42
).sort_values()

3483      ANDERSON GWEN V & TD, ML, LD, NL
150     ANTISDALE THOMAS & CRANE ELIZABETH
4648     BENHAM MATHEW & VEGA HEATHER (LC)
3475    BLADES GEORGE/AMES MICHAEL & KIMMI
3662             BOON MORGAN/DESAI BANSILI
                       ...                
3672     VIR KAUR JATINDER/SINGH HARWINDER
267             WARD, KASEY & TERRY JOE II
452     WEELDREYER ANDIS JON & ADRIAN LOIS
767     WERTKIN, SUSAN GATES & ROBERT ALAN
7304             WITT DAVID JC & MENDY RAE
Name: entity_name, Length: 100, dtype: str

In [15]:
joint_summary = pd.Series({
    "Contains &": joint_names.str.contains(
        "&",
        regex=False
    ).sum(),

    "Contains AND": joint_names.str.contains(
        r"\bAND\b",
        regex=True
    ).sum(),

    "Contains /": joint_names.str.contains(
        "/",
        regex=False
    ).sum(),

    "Contains comma": joint_names.str.contains(
        ",",
        regex=False
    ).sum(),
})

joint_summary

Contains &        180
Contains AND        7
Contains /         26
Contains comma     99
dtype: int64

## Inspecting Joint type entities

In [16]:
joint_names = (
    entity_occurrences.loc[
        entity_occurrences["entity_type"] == "Joint",
        "entity_name"
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

def profile_joint_name(name):

    candidate = resolver._normalize_entity_type_candidate(name)

    return pd.Series({
        "ampersand_count": candidate.count("&"),
        "semicolon_count": candidate.count(";"),
        "slash_count": candidate.count("/"),

        "and_count": len(
            re.findall(
                r"\bAND\b",
                candidate
            )
        ),

        "comma_count": candidate.count(","),

        "contains_et_al": bool(
            re.search(
                r"\bET AL\b",
                candidate
            )
        ),

        "token_count": len(
            candidate.split()
        )
    })

joint_profile = (
    joint_names
    .to_frame(name="entity_name")
)

joint_profile = pd.concat(
    [
        joint_profile,

        joint_profile["entity_name"]
        .apply(profile_joint_name)
    ],
    axis=1
)

joint_profile.head()

,entity_name,ampersand_count,semicolon_count,slash_count,and_count,comma_count,contains_et_al,token_count
0,CLAWSON DORRINDA &,1,0,0,0,0,False,3
1,SHILTS NANCY & MCCOY THOMAS,1,0,0,0,0,False,5
2,PAK SAM K & KANG IM KYO,1,0,0,0,0,False,7
3,GRAY MARK & MCCOOK MELISSA,1,0,0,0,0,False,5
4,ALEXANDER F & PAVELEC J & PAFF D L,2,0,0,0,0,False,9


In [17]:
joint_profile["separator_type"] = "Other"

joint_profile.loc[
    (
        joint_profile["ampersand_count"] == 1
    )
    &
    (
        joint_profile[
            [
                "semicolon_count",
                "slash_count",
                "and_count"
            ]
        ].sum(axis=1) == 0
    ),
    "separator_type"
] = "Single ampersand"

joint_profile.loc[
    joint_profile["semicolon_count"] > 0,
    "separator_type"
] = "Semicolon"

joint_profile.loc[
    joint_profile["slash_count"] > 0,
    "separator_type"
] = "Slash"

joint_profile.loc[
    joint_profile["and_count"] > 0,
    "separator_type"
] = "AND"

joint_profile.loc[
    (
        joint_profile[
            [
                "ampersand_count",
                "semicolon_count",
                "slash_count",
                "and_count"
            ]
        ].sum(axis=1) > 1
    ),
    "separator_type"
] = "Multiple separators"

joint_profile["separator_type"].value_counts()

separator_type
Single ampersand       152
Multiple separators     31
Slash                   15
AND                      6
Semicolon                3
Name: count, dtype: int64

In [18]:
simple_ampersand = (
    joint_profile[
        (
            joint_profile["separator_type"]
            == "Single ampersand"
        )
        &
        (~joint_profile["contains_et_al"])
    ]
)

print(
    f"Unique simple ampersand Joint names: "
    f"{len(simple_ampersand):,}"
)

simple_ampersand[
    [
        "entity_name",
        "comma_count",
        "token_count"
    ]
].sample(
    min(100, len(simple_ampersand)),
    random_state=42
)

Unique simple ampersand Joint names: 146


,entity_name,comma_count,token_count
57,MICHAEL L. & SARAH D. CANTRELL,0,6
144,FINK JOHN & HUFF MELLISA J,0,6
31,WEELDREYER ANDIS JON & ADRIAN LOIS,0,6
22,GLEASON GEORGE & BOWERS AMY,0,5
52,"TRACY, ELLEN A & RUTH SHEPPARD",1,6
...,...,...,...
20,ROMACK DOUGLAS L & BRADSHAW SUSANNE,0,6
43,"CORONADO, FIDEL & ORTA IVONE",1,5
98,STANFILL NICOLE M & SALTER CHARLES,0,6
174,FOX JOSE & PIER STEPHANIE (LC),0,5


In [19]:
simple_ampersand["comma_count"].value_counts().sort_index()

comma_count
0    81
1    61
2     3
3     1
Name: count, dtype: int64

In [20]:
simple_ampersand.groupby(
    "comma_count"
)["token_count"].describe()

,count,mean,std,min,25%,50%,75%,max
comma_count,,,,,,,,
0,81.0,5.358025,0.884712,3.0,5.0,5.0,6.0,7.0
1,61.0,5.442623,0.940294,3.0,5.0,6.0,6.0,7.0
2,3.0,5.666667,1.154701,5.0,5.0,5.0,6.0,7.0
3,1.0,8.000000,NaN,8.0,8.0,8.0,8.0,8.0


In [21]:
print(len(simple_ampersand))

print(
    simple_ampersand["comma_count"]
    .isna()
    .sum()
)

146
0


In [22]:
medium_joint_candidates = (
    entity_occurrences[
        (entity_occurrences["entity_type"] == "Joint")
        &
        (
            entity_occurrences["joint_parse_confidence"]
            == "Medium"
        )
    ]
    .copy()
)

print(
    "Medium-confidence Joint occurrences:",
    f"{len(medium_joint_candidates):,}"
)

print(
    "Unique Medium-confidence Joint names:",
    f"{medium_joint_candidates['entity_name'].nunique():,}"
)

Medium-confidence Joint occurrences: 0
Unique Medium-confidence Joint names: 0


In [23]:
medium_joint_candidates[
    "joint_parse_rule"
].value_counts()

Series([], Name: count, dtype: int64)

In [24]:
medium_joint_names = (
    medium_joint_candidates[
        [
            "entity_name",
            "joint_parse_rule"
        ]
    ]
    .drop_duplicates()
)

medium_joint_names.sample(
    min(100, len(medium_joint_names)),
    random_state=42
).sort_values(
    "entity_name"
)

,entity_name,joint_parse_rule


In [25]:
expanded_joint_audit = (
    entity_occurrences[
        entity_occurrences["joint_expanded"]
    ]
    [
        [
            "source_occurrence_id",
            "source_entity_name",
            "entity_name",
            "joint_party_index",
            "joint_party_count",
            "joint_parse_rule",
            "joint_parse_confidence"
        ]
    ]
    .sort_values(
        [
            "source_occurrence_id",
            "joint_party_index"
        ]
    )
)

expanded_joint_audit.head(100)

,source_occurrence_id,source_entity_name,entity_name,joint_party_index,joint_party_count,joint_parse_rule,joint_parse_confidence
4,5,SPIEKERMANN TIM & JUDY,"SPIEKERMANN, TIM",1.0,2.0,shared_surname_single_right_name,High
5,5,SPIEKERMANN TIM & JUDY,"SPIEKERMANN, JUDY",2.0,2.0,shared_surname_single_right_name,High
6,6,LINDBERG MARK H & LYNNETTE,"LINDBERG, MARK H",1.0,2.0,shared_surname_single_right_name,High
7,6,LINDBERG MARK H & LYNNETTE,"LINDBERG, LYNNETTE",2.0,2.0,shared_surname_single_right_name,High
18,17,THOMSON STEVEN L & ASHLEY A,"THOMSON, STEVEN L",1.0,2.0,shared_surname_initial_no_comma,Medium
...,...,...,...,...,...,...,...
245,198,"KYES-MYLAND, CONNIE & WHISLER, VICKI","WHISLER, VICKI",2.0,2.0,explicit_surnames_comma,High
246,199,"LONEY, MARY & STEVE","LONEY, MARY",1.0,2.0,shared_surname_comma,High
247,199,"LONEY, MARY & STEVE","LONEY, STEVE",2.0,2.0,shared_surname_comma,High
253,205,"KORTZ, TAYLOR & DZIEWICKI, GREGORY","KORTZ, TAYLOR",1.0,2.0,explicit_surnames_comma,High


In [26]:
expanded_joint_pairs = (
    expanded_joint_audit
    .pivot_table(
        index=[
            "source_occurrence_id",
            "source_entity_name",
            "joint_parse_rule",
            "joint_parse_confidence"
        ],
        columns="joint_party_index",
        values="entity_name",
        aggfunc="first"
    )
    .reset_index()
    .rename(
        columns={
            1: "person_1",
            2: "person_2"
        }
    )
)

medium_joint_names.sort_values(
    "entity_name"
).reset_index(
    drop=True
)

,entity_name,joint_parse_rule


In [27]:
expanded_counts = (
    entity_occurrences[
        entity_occurrences["joint_expanded"]
    ]
    .groupby(
        "source_occurrence_id"
    )
    .size()
)

print(
    "Expanded source occurrences not producing exactly 2 rows:",
    (expanded_counts != 2).sum()
)

Expanded source occurrences not producing exactly 2 rows: 0


In [28]:
expanded_source_ids = set(
    entity_occurrences.loc[
        entity_occurrences["joint_expanded"],
        "source_occurrence_id"
    ]
)

surviving_joint_rows = (
    entity_occurrences[
        entity_occurrences["source_occurrence_id"]
        .isin(expanded_source_ids)
        &
        (
            entity_occurrences["entity_type"]
            == "Joint"
        )
    ]
)

print(
    "Expanded sources still containing a Joint row:",
    len(surviving_joint_rows)
)

Expanded sources still containing a Joint row: 0


In [29]:
entity_occurrences["entity_type"].value_counts()

entity_type
Person          9589
Organization    1978
Joint            469
Unknown           22
Name: count, dtype: int64

## Analyzing Joint type entities after splitting high and medium confidence one and two comma Joint entities

In [30]:
remaining_joint = (
    entity_occurrences[
        entity_occurrences["entity_type"] == "Joint"
    ]
    .copy()
)

print(
    "Remaining Joint occurrences:",
    f"{len(remaining_joint):,}"
)

print(
    "Unique remaining Joint names:",
    f"{remaining_joint['entity_name'].nunique():,}"
)

Remaining Joint occurrences: 469
Unique remaining Joint names: 207


In [31]:
remaining_joint_names = (
    remaining_joint["entity_name"]
    .drop_duplicates()
    .reset_index(drop=True)
)

In [32]:
remaining_parseable = (
    remaining_joint_names
    .apply(
        lambda name:
        resolver._parse_joint_name(name)
        is not None
    )
)

print(
    "Remaining Joint names still parseable "
    "by current parser:",
    remaining_parseable.sum()
)

Remaining Joint names still parseable by current parser: 0


In [33]:
def profile_joint_name(name):

    candidate = (
        resolver
        ._normalize_entity_type_candidate(name)
    )

    return pd.Series({
        "ampersand_count":
            candidate.count("&"),

        "semicolon_count":
            candidate.count(";"),

        "slash_count":
            candidate.count("/"),

        "and_count":
            len(
                re.findall(
                    r"\bAND\b",
                    candidate
                )
            ),

        "comma_count":
            candidate.count(","),

        "contains_et_al":
            bool(
                re.search(
                    r"\bET AL\b",
                    candidate
                )
            ),

        "token_count":
            len(candidate.split())
    })

In [34]:
remaining_joint_profile = (
    remaining_joint_names
    .to_frame(name="entity_name")
)

remaining_joint_profile = pd.concat(
    [
        remaining_joint_profile,

        remaining_joint_profile[
            "entity_name"
        ].apply(
            profile_joint_name
        )
    ],
    axis=1
)

In [35]:
remaining_joint_profile[
    "comma_count"
].value_counts().sort_index()

comma_count
0    108
1     81
2     17
3      1
Name: count, dtype: int64

In [36]:
remaining_joint_profile[
    [
        "ampersand_count",
        "semicolon_count",
        "slash_count",
        "and_count"
    ]
].describe()

,ampersand_count,semicolon_count,slash_count,and_count
count,207.000000,207.000000,207.000000,207.000000
mean,1.000000,0.033816,0.120773,0.033816
std,0.582929,0.206253,0.355133,0.181195
min,0.000000,0.000000,0.000000,0.000000
25%,1.000000,0.000000,0.000000,0.000000
50%,1.000000,0.000000,0.000000,0.000000
75%,1.000000,0.000000,0.000000,0.000000
max,4.000000,2.000000,2.000000,1.000000


In [37]:
zero_comma_single_ampersand = (
    remaining_joint_profile[
        (remaining_joint_profile["ampersand_count"] == 1)
        &
        (remaining_joint_profile["comma_count"] == 0)
        &
        (remaining_joint_profile["semicolon_count"] == 0)
        &
        (remaining_joint_profile["slash_count"] == 0)
        &
        (remaining_joint_profile["and_count"] == 0)
        &
        (~remaining_joint_profile["contains_et_al"])
    ]
    .copy()
)

print(
    "Unique zero-comma single-ampersand names:",
    f"{len(zero_comma_single_ampersand):,}"
)

Unique zero-comma single-ampersand names: 81


In [38]:
zero_comma_single_ampersand[
    [
        "entity_name",
        "token_count"
    ]
].sample(
    min(
        100,
        len(zero_comma_single_ampersand)
    ),
    random_state=42
).sort_values(
    "entity_name"
)

,entity_name,token_count
173,& FALES SHERRIE,3
183,AARON DANIEL & AABERG GAYE,5
10,AHLRICH JANITH L &,4
110,ALLEN JENNIFER & RICHTER RUSSELL E,6
12,ANTISDALE THOMAS & CRANE ELIZABETH,5
...,...,...
8,VILLANUEVA MIGUEL &,3
21,WADINGTON LOREN A & DAVID L TRSTEES,7
31,WEELDREYER ANDIS JON & ADRIAN LOIS,6
185,WILLIAMSON CHARLES & CLARK NADINE,5


## Analyzing single ampersand cases with no commas

In [39]:
def profile_zero_comma_joint(name):

    candidate = (
        resolver
        ._normalize_entity_type_candidate(name)
    )

    left, right = [
        part.strip()
        for part in candidate.split("&", 1)
    ]

    left_tokens = left.split()
    right_tokens = right.split()

    return pd.Series({
        "left": left,
        "right": right,

        "left_token_count":
            len(left_tokens),

        "right_token_count":
            len(right_tokens),

        "left_empty":
            len(left_tokens) == 0,

        "right_empty":
            len(right_tokens) == 0,

        "left_first_token":
            left_tokens[0]
            if left_tokens
            else "",

        "right_first_token":
            right_tokens[0]
            if right_tokens
            else "",

        "right_last_token":
            right_tokens[-1]
            if right_tokens
            else "",

        "right_ends_initial":
            bool(
                right_tokens
                and len(right_tokens[-1]) == 1
            ),

        "same_leading_token":
            bool(
                left_tokens
                and right_tokens
                and left_tokens[0]
                == right_tokens[0]
            )
    })


zero_comma_structure = pd.concat(
    [
        zero_comma_single_ampersand[
            ["entity_name"]
        ].reset_index(drop=True),

        zero_comma_single_ampersand[
            "entity_name"
        ]
        .reset_index(drop=True)
        .apply(profile_zero_comma_joint)
    ],
    axis=1
)

In [40]:
zero_comma_structure[
    "right_token_count"
].value_counts().sort_index()

right_token_count
0     7
2    58
3    16
Name: count, dtype: int64

In [41]:
pd.crosstab(
    zero_comma_structure[
        "right_token_count"
    ],
    zero_comma_structure[
        "right_ends_initial"
    ]
)

right_ends_initial,False,True
right_token_count,,
0,7,0
2,57,1
3,8,8


In [42]:
print(
    "Trailing-empty records:",
    zero_comma_structure[
        "right_empty"
    ].sum()
)

print(
    "Same leading token on both sides:",
    zero_comma_structure[
        "same_leading_token"
    ].sum()
)

Trailing-empty records: 7
Same leading token on both sides: 0


### Continuing analysis

In [43]:
pd.crosstab(
    zero_comma_structure["left_token_count"],
    zero_comma_structure["right_token_count"]
)

left_one_token = (
    zero_comma_structure[
        (zero_comma_structure["left_token_count"] == 1)
        &
        (~zero_comma_structure["right_empty"])
    ]
    .copy()
)

print(
    "Unique names with one token on the left:",
    len(left_one_token)
)

left_one_token[
    [
        "entity_name",
        "left",
        "right",
        "right_token_count"
    ]
].sort_values(
    "entity_name"
)

Unique names with one token on the left: 1


,entity_name,left,right,right_token_count
74,C/O SCOFIELD & VAN DENBELDT,SCOFIELD,VAN DENBELDT,2


In [44]:
right_two_ending_initial = (
    zero_comma_structure[
        (zero_comma_structure["right_token_count"] == 2)
        &
        (zero_comma_structure["right_ends_initial"])
        &
        (~zero_comma_structure["right_empty"])
    ]
    .copy()
)

right_two_ending_initial[
    [
        "entity_name",
        "left",
        "right",
        "left_token_count"
    ]
].sort_values(
    "entity_name"
)

,entity_name,left,right,left_token_count
34,CAMPBELL J SR & L M,CAMPBELL J SR,L M,3


In [45]:
right_two_full_tokens = (
    zero_comma_structure[
        (zero_comma_structure["right_token_count"] == 2)
        &
        (~zero_comma_structure["right_ends_initial"])
    ]
    .copy()
)

right_two_full_tokens[
    [
        "entity_name",
        "left",
        "right",
        "left_token_count"
    ]
].sort_values(
    "entity_name"
)

,entity_name,left,right,left_token_count
65,& FALES SHERRIE,,FALES SHERRIE,0
71,AARON DANIEL & AABERG GAYE,AARON DANIEL,AABERG GAYE,2
11,ANTISDALE THOMAS & CRANE ELIZABETH,ANTISDALE THOMAS,CRANE ELIZABETH,2
27,BAIRD ROBERT & MARY ANNE,BAIRD ROBERT,MARY ANNE,2
26,BAIRD ROBERT R & MARY ANNE,BAIRD ROBERT R,MARY ANNE,3
21,BELL ERIK & MARBLE MORGAN,BELL ERIK,MARBLE MORGAN,2
67,BENHAM MATHEW & VEGA HEATHER (LC),BENHAM MATHEW,VEGA HEATHER,2
58,BLOK BRENNON & JONGEKRYG ABAGAIL,BLOK BRENNON,JONGEKRYG ABAGAIL,2
42,BRIERLY ROBERT & STREET DANIEL,BRIERLY ROBERT,STREET DANIEL,2
13,BUCHHOLZ ROGER & DELAYNE CHARVAT,BUCHHOLZ ROGER,DELAYNE CHARVAT,2


In [46]:
entity_occurrences.loc[
    entity_occurrences["joint_expanded"],
    "joint_parse_rule"
].value_counts()

joint_parse_rule
shared_surname_comma                   1716
shared_surname_single_right_name        752
shared_surname_initial_no_comma         560
explicit_surnames_comma                 480
explicit_shared_surname_no_comma         54
shared_trailing_surname_no_comma         10
explicit_surnames_initials_no_comma       4
Name: count, dtype: int64

## Investigating building a name dictionary from county data

In [47]:
known_first_names = set()
known_last_names = set()

for name in (
    entity_occurrences.loc[
        entity_occurrences["entity_type"] == "Person",
        "entity_name"
    ]
    .drop_duplicates()
):

    normalized = resolver._normalize_match_name(
        name
    )

    # Only use comma-format names for building the vocabulary.
    # Their name orientation is explicitly represented in the
    # source rather than inferred.
    if "," not in normalized:
        continue

    parsed = resolver._parse_person_name(
        normalized
    )

    if parsed is None:
        continue

    known_first_names.add(
        parsed["first_name"]
    )

    known_last_names.add(
        parsed["last_name"]
    )

In [48]:
def get_name_position_evidence(row):

    right_tokens = row["right"].split()

    if len(right_tokens) != 2:
        return pd.Series({
            "right_token_1": "",
            "right_token_2": "",
            "token_1_seen_first": False,
            "token_1_seen_last": False,
            "token_2_seen_first": False,
            "token_2_seen_last": False
        })

    token_1, token_2 = right_tokens

    return pd.Series({
        "right_token_1": token_1,
        "right_token_2": token_2,

        "token_1_seen_first":
            token_1 in known_first_names,

        "token_1_seen_last":
            token_1 in known_last_names,

        "token_2_seen_first":
            token_2 in known_first_names,

        "token_2_seen_last":
            token_2 in known_last_names
    })


right_two_full_evidence = pd.concat(
    [
        right_two_full_tokens.reset_index(
            drop=True
        ),

        right_two_full_tokens.reset_index(
            drop=True
        ).apply(
            get_name_position_evidence,
            axis=1
        )
    ],
    axis=1
)

In [49]:
def classify_right_side_evidence(row):

    # Strong evidence for:
    #
    # SURNAME FIRST
    #
    # e.g. MCCADEN SHAWNA
    if (
        row["token_1_seen_last"]
        and row["token_2_seen_first"]
        and not row["token_1_seen_first"]
    ):
        return "Likely separate surname"

    # Strong evidence for:
    #
    # FIRST MIDDLE
    #
    # e.g. MARY ANNE
    if (
        row["token_1_seen_first"]
        and not row["token_1_seen_last"]
        and not row["token_2_seen_last"]
    ):
        return "Likely shared surname"

    return "Ambiguous"


right_two_full_evidence[
    "name_evidence"
] = right_two_full_evidence.apply(
    classify_right_side_evidence,
    axis=1
)

In [50]:
right_two_full_evidence[
    "name_evidence"
].value_counts()

name_evidence
Ambiguous                  36
Likely separate surname    11
Likely shared surname      10
Name: count, dtype: int64

In [51]:
right_two_full_evidence[
    [
        "entity_name",
        "right",
        "right_token_1",
        "right_token_2",
        "token_1_seen_first",
        "token_1_seen_last",
        "token_2_seen_first",
        "token_2_seen_last",
        "name_evidence"
    ]
].sort_values(
    [
        "name_evidence",
        "entity_name"
    ]
)

,entity_name,right,right_token_1,right_token_2,token_1_seen_first,token_1_seen_last,token_2_seen_first,token_2_seen_last,name_evidence
42,& FALES SHERRIE,FALES SHERRIE,FALES,SHERRIE,False,False,True,False,Ambiguous
47,AARON DANIEL & AABERG GAYE,AABERG GAYE,AABERG,GAYE,False,False,False,False,Ambiguous
4,ANTISDALE THOMAS & CRANE ELIZABETH,CRANE ELIZABETH,CRANE,ELIZABETH,False,False,True,False,Ambiguous
38,BLOK BRENNON & JONGEKRYG ABAGAIL,JONGEKRYG ABAGAIL,JONGEKRYG,ABAGAIL,False,False,False,False,Ambiguous
27,BRIERLY ROBERT & STREET DANIEL,STREET DANIEL,STREET,DANIEL,False,False,True,True,Ambiguous
6,BUCHHOLZ ROGER & DELAYNE CHARVAT,DELAYNE CHARVAT,DELAYNE,CHARVAT,False,False,False,False,Ambiguous
20,BUSTER WILLIAM & MCCADEN SHAWNA,MCCADEN SHAWNA,MCCADEN,SHAWNA,False,False,False,False,Ambiguous
52,C/O PR DEBRA BRITTEN & STEVEN HUNT,STEVEN HUNT,STEVEN,HUNT,True,False,False,True,Ambiguous
50,C/O SCOFIELD & VAN DENBELDT,VAN DENBELDT,VAN,DENBELDT,False,True,False,False,Ambiguous
46,C/O: HARJINDER KAUR & RAJEET KAUR,RAJEET KAUR,RAJEET,KAUR,False,False,False,True,Ambiguous


### Adding address data

In [52]:
person_evidence_rows = []

for _, row in entity_occurrences.iterrows():

    # Avoid using Joint-derived people to prove our own Joint
    # decomposition assumptions.
    if row["entity_type"] != "Person":
        continue

    if row.get("joint_expanded", False):
        continue

    parsed = resolver._parse_person_name(
        row["entity_name"]
    )

    if parsed is None:
        continue

    person_evidence_rows.append({
        "entity_name": row["entity_name"],
        "associated_address": row["associated_address"],
        "last_name": parsed["last_name"],
        "first_name": parsed["first_name"],
        "middle_tokens": parsed["middle_tokens"],
        "suffix": parsed["suffix"]
    })


person_evidence = pd.DataFrame(
    person_evidence_rows
)

In [53]:
def evaluate_two_full_token_joint(row):

    left_tokens = row["left"].split()
    right_tokens = row["right"].split()

    result = {
        "shared_candidate": "",
        "separate_candidate": "",
        "shared_exact_anywhere": False,
        "separate_exact_anywhere": False,
        "shared_same_address": False,
        "separate_same_address": False
    }

    if (
        len(left_tokens) < 2
        or len(right_tokens) != 2
    ):
        return pd.Series(result)

    shared_surname = left_tokens[0]

    right_1 = right_tokens[0]
    right_2 = right_tokens[1]

    # Hypothesis A:
    # right side is FIRST MIDDLE and inherits left surname.
    shared_candidate = (
        f"{shared_surname}, "
        f"{right_1} {right_2}"
    )

    # Hypothesis B:
    # right side is LAST FIRST.
    separate_candidate = (
        f"{right_1}, "
        f"{right_2}"
    )

    result[
        "shared_candidate"
    ] = shared_candidate

    result[
        "separate_candidate"
    ] = separate_candidate

    shared_parsed = (
        resolver._parse_person_name(
            shared_candidate
        )
    )

    separate_parsed = (
        resolver._parse_person_name(
            separate_candidate
        )
    )

    address = row.get(
        "associated_address",
        ""
    )

    if shared_parsed is not None:

        shared_matches = person_evidence[
            (person_evidence["last_name"]
             == shared_parsed["last_name"])
            &
            (person_evidence["first_name"]
             == shared_parsed["first_name"])
            &
            (person_evidence["middle_tokens"]
             == shared_parsed["middle_tokens"])
        ]

        result[
            "shared_exact_anywhere"
        ] = len(shared_matches) > 0

        result[
            "shared_same_address"
        ] = (
            (
                shared_matches[
                    "associated_address"
                ] == address
            ).any()
            if address
            else False
        )

    if separate_parsed is not None:

        separate_matches = person_evidence[
            (person_evidence["last_name"]
             == separate_parsed["last_name"])
            &
            (person_evidence["first_name"]
             == separate_parsed["first_name"])
            &
            (person_evidence["middle_tokens"]
             == separate_parsed["middle_tokens"])
        ]

        result[
            "separate_exact_anywhere"
        ] = len(separate_matches) > 0

        result[
            "separate_same_address"
        ] = (
            (
                separate_matches[
                    "associated_address"
                ] == address
            ).any()
            if address
            else False
        )

    return pd.Series(result)

In [54]:
right_two_full_with_address = (
    right_two_full_tokens
    .merge(
        remaining_joint[
            [
                "entity_name",
                "associated_address"
            ]
        ].drop_duplicates(),
        on="entity_name",
        how="left"
    )
)

right_two_full_with_address

,entity_name,left,right,left_token_count,right_token_count,left_empty,right_empty,left_first_token,right_first_token,right_last_token,right_ends_initial,same_leading_token,associated_address
0,SHILTS NANCY & MCCOY THOMAS,SHILTS NANCY,MCCOY THOMAS,2,2,False,False,SHILTS,MCCOY,THOMAS,False,False,"8881 W C AVE, KALAMAZOO, MI 49009"
1,SHILTS NANCY & MCCOY THOMAS,SHILTS NANCY,MCCOY THOMAS,2,2,False,False,SHILTS,MCCOY,THOMAS,False,False,
2,GRAY MARK & MCCOOK MELISSA,GRAY MARK,MCCOOK MELISSA,2,2,False,False,GRAY,MCCOOK,MELISSA,False,False,"6708 OWEN DR, KALAMAZOO, MI 49009"
3,GRAY MARK & MCCOOK MELISSA,GRAY MARK,MCCOOK MELISSA,2,2,False,False,GRAY,MCCOOK,MELISSA,False,False,
4,PLOWMAN GLENN W & JUDITH ETA,PLOWMAN GLENN W,JUDITH ETA,3,2,False,False,PLOWMAN,JUDITH,ETA,False,False,"3504 W D AVE, KALAMAZOO, MI 49009"
...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,C/O PR DEBRA BRITTEN & STEVEN HUNT,PR DEBRA BRITTEN,STEVEN HUNT,3,2,False,False,PR,STEVEN,HUNT,False,False,"1369 M DR S, E LEROY, MI 49051"
92,PLOWMAN GLENN W & JUDITH ET,PLOWMAN GLENN W,JUDITH ET,3,2,False,False,PLOWMAN,JUDITH,ET,False,False,"3504 W D AVE, KALAMAZOO, MI 49009"
93,GATES MARIAN & KELLOGG HARR,GATES MARIAN,KELLOGG HARR,2,2,False,False,GATES,KELLOGG,HARR,False,False,"5461 COLLINGWOOD, KALAMAZOO, MI 49004"
94,NEFF JR KENNETH & FAUBLE TIFFANY,NEFF JR KENNETH,FAUBLE TIFFANY,3,2,False,False,NEFF,FAUBLE,TIFFANY,False,False,"6233 E W AVE, VICKSBURG, MI 49097"


In [55]:
context_evidence = pd.concat(
    [
        right_two_full_with_address
        .reset_index(drop=True),

        right_two_full_with_address
        .reset_index(drop=True)
        .apply(
            evaluate_two_full_token_joint,
            axis=1
        )
    ],
    axis=1
)

context_evidence

,entity_name,left,right,left_token_count,right_token_count,left_empty,right_empty,left_first_token,right_first_token,right_last_token,right_ends_initial,same_leading_token,associated_address,shared_candidate,separate_candidate,shared_exact_anywhere,separate_exact_anywhere,shared_same_address,separate_same_address
0,SHILTS NANCY & MCCOY THOMAS,SHILTS NANCY,MCCOY THOMAS,2,2,False,False,SHILTS,MCCOY,THOMAS,False,False,"8881 W C AVE, KALAMAZOO, MI 49009","SHILTS, MCCOY THOMAS","MCCOY, THOMAS",False,False,False,False
1,SHILTS NANCY & MCCOY THOMAS,SHILTS NANCY,MCCOY THOMAS,2,2,False,False,SHILTS,MCCOY,THOMAS,False,False,,"SHILTS, MCCOY THOMAS","MCCOY, THOMAS",False,False,False,False
2,GRAY MARK & MCCOOK MELISSA,GRAY MARK,MCCOOK MELISSA,2,2,False,False,GRAY,MCCOOK,MELISSA,False,False,"6708 OWEN DR, KALAMAZOO, MI 49009","GRAY, MCCOOK MELISSA","MCCOOK, MELISSA",False,False,False,False
3,GRAY MARK & MCCOOK MELISSA,GRAY MARK,MCCOOK MELISSA,2,2,False,False,GRAY,MCCOOK,MELISSA,False,False,,"GRAY, MCCOOK MELISSA","MCCOOK, MELISSA",False,False,False,False
4,PLOWMAN GLENN W & JUDITH ETA,PLOWMAN GLENN W,JUDITH ETA,3,2,False,False,PLOWMAN,JUDITH,ETA,False,False,"3504 W D AVE, KALAMAZOO, MI 49009","PLOWMAN, JUDITH ETA","JUDITH, ETA",False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91,C/O PR DEBRA BRITTEN & STEVEN HUNT,PR DEBRA BRITTEN,STEVEN HUNT,3,2,False,False,PR,STEVEN,HUNT,False,False,"1369 M DR S, E LEROY, MI 49051","PR, STEVEN HUNT","STEVEN, HUNT",False,False,False,False
92,PLOWMAN GLENN W & JUDITH ET,PLOWMAN GLENN W,JUDITH ET,3,2,False,False,PLOWMAN,JUDITH,ET,False,False,"3504 W D AVE, KALAMAZOO, MI 49009","PLOWMAN, JUDITH ET","JUDITH, ET",False,False,False,False
93,GATES MARIAN & KELLOGG HARR,GATES MARIAN,KELLOGG HARR,2,2,False,False,GATES,KELLOGG,HARR,False,False,"5461 COLLINGWOOD, KALAMAZOO, MI 49004","GATES, KELLOGG HARR","KELLOGG, HARR",False,False,False,False
94,NEFF JR KENNETH & FAUBLE TIFFANY,NEFF JR KENNETH,FAUBLE TIFFANY,3,2,False,False,NEFF,FAUBLE,TIFFANY,False,False,"6233 E W AVE, VICKSBURG, MI 49097","NEFF, FAUBLE TIFFANY","FAUBLE, TIFFANY",False,False,False,False


In [56]:
context_evidence_by_name = (
    context_evidence
    .groupby(
        "entity_name",
        as_index=False
    )
    .agg(
        shared_candidate=(
            "shared_candidate",
            "first"
        ),
        separate_candidate=(
            "separate_candidate",
            "first"
        ),

        shared_exact_anywhere=(
            "shared_exact_anywhere",
            "any"
        ),
        separate_exact_anywhere=(
            "separate_exact_anywhere",
            "any"
        ),

        shared_same_address=(
            "shared_same_address",
            "any"
        ),
        separate_same_address=(
            "separate_same_address",
            "any"
        ),

        address_count=(
            "associated_address",
            "nunique"
        )
    )
)

In [57]:
print(
    "Unique Joint names in contextual analysis:",
    len(context_evidence_by_name)
)

Unique Joint names in contextual analysis: 57


In [58]:
pd.crosstab(
    context_evidence_by_name[
        "shared_same_address"
    ],
    context_evidence_by_name[
        "separate_same_address"
    ],
    margins=True
)

separate_same_address,False,All
shared_same_address,,
False,57,57
All,57,57


In [59]:
same_address_evidence = (
    context_evidence_by_name[
        context_evidence_by_name[
            [
                "shared_same_address",
                "separate_same_address"
            ]
        ].any(axis=1)
    ]
    [
        [
            "entity_name",
            "shared_candidate",
            "separate_candidate",
            "shared_same_address",
            "separate_same_address",
            "shared_exact_anywhere",
            "separate_exact_anywhere",
            "address_count"
        ]
    ]
    .sort_values(
        [
            "shared_same_address",
            "separate_same_address",
            "entity_name"
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
)

same_address_evidence

,entity_name,shared_candidate,separate_candidate,shared_same_address,separate_same_address,shared_exact_anywhere,separate_exact_anywhere,address_count


In [60]:
def classify_contextual_evidence(row):

    shared_same = row[
        "shared_same_address"
    ]

    separate_same = row[
        "separate_same_address"
    ]

    shared_any = row[
        "shared_exact_anywhere"
    ]

    separate_any = row[
        "separate_exact_anywhere"
    ]


    # ------------------------------------------------------
    # Strongest evidence: same address
    # ------------------------------------------------------

    if shared_same and not separate_same:
        return "Shared surname - same address"

    if separate_same and not shared_same:
        return "Separate surnames - same address"

    if shared_same and separate_same:
        return "Conflict - both same address"


    # ------------------------------------------------------
    # Secondary evidence: exact candidate elsewhere
    # ------------------------------------------------------

    if shared_any and not separate_any:
        return "Shared surname - elsewhere"

    if separate_any and not shared_any:
        return "Separate surnames - elsewhere"

    if shared_any and separate_any:
        return "Conflict - both elsewhere"


    # ------------------------------------------------------
    # No contextual evidence
    # ------------------------------------------------------

    return "No contextual evidence"

In [61]:
context_evidence_by_name[
    "context_decision"
] = (
    context_evidence_by_name
    .apply(
        classify_contextual_evidence,
        axis=1
    )
)

In [62]:
context_evidence_by_name[
    "context_decision"
].value_counts()

context_decision
No contextual evidence    57
Name: count, dtype: int64

## Addressing cases of same name similar addresses

In [63]:
STREET_SUFFIXES = {
    "ST",
    "AVE",
    "RD",
    "DR",
    "BLVD",
    "LN",
    "CT",
    "CIR",
    "PL",
    "PKWY",
    "TER",
    "HWY",
    "WAY",
    "TRL"
}

UNIT_MARKERS = {
    "APT",
    "STE",
    "UNIT",
    "BLDG",
    "FL",
    "RM"
}


def parse_address_suffix_for_audit(address):
    """
    Break a cleaned address into components needed to detect
    cases where a street suffix is present in one representation
    and absent in another.

    This is for analysis only and does not modify the address.
    """

    empty_result = pd.Series({
        "street_base": None,
        "street_suffix": None,
        "unit": None,
        "locality": None
    })

    if pd.isna(address):
        return empty_result

    address = str(address).strip()

    if not address:
        return empty_result

    parts = [
        part.strip()
        for part in address.split(",")
    ]

    # Require at least:
    #
    # street, locality...
    if len(parts) < 2:
        return empty_result

    street_line = parts[0]

    # Don't try to interpret PO Boxes as street addresses.
    if street_line.startswith("PO BOX"):
        return empty_result

    locality = ", ".join(parts[1:])

    tokens = street_line.split()

    if len(tokens) < 2:
        return empty_result


    # ----------------------------------------------------------
    # Separate unit information
    # ----------------------------------------------------------

    unit_index = None

    for i, token in enumerate(tokens):

        if (
            token in UNIT_MARKERS
            or token.startswith("#")
        ):
            unit_index = i
            break

    if unit_index is not None:

        street_tokens = tokens[:unit_index]
        unit_tokens = tokens[unit_index:]

    else:

        street_tokens = tokens
        unit_tokens = []


    if len(street_tokens) < 2:
        return empty_result


    # Require something resembling a street number.
    if not any(
        character.isdigit()
        for character in street_tokens[0]
    ):
        return empty_result


    # ----------------------------------------------------------
    # Identify street suffix
    # ----------------------------------------------------------

    if street_tokens[-1] in STREET_SUFFIXES:

        street_suffix = street_tokens[-1]

        street_base_tokens = (
            street_tokens[:-1]
        )

    else:

        street_suffix = ""

        street_base_tokens = street_tokens


    if not street_base_tokens:
        return empty_result


    return pd.Series({
        "street_base":
            " ".join(street_base_tokens),

        "street_suffix":
            street_suffix,

        "unit":
            " ".join(unit_tokens),

        "locality":
            locality
    })

In [64]:
address_suffix_audit = (
    entities[
        entities["canonical_address"]
        .fillna("")
        .str.strip()
        .ne("")
    ]
    .copy()
)

parsed_addresses = (
    address_suffix_audit[
        "canonical_address"
    ]
    .apply(
        parse_address_suffix_for_audit
    )
)

address_suffix_audit = pd.concat(
    [
        address_suffix_audit.reset_index(
            drop=True
        ),

        parsed_addresses.reset_index(
            drop=True
        )
    ],
    axis=1
)

In [65]:
address_suffix_candidates = []

group_columns = [
    "entity_type",
    "canonical_name",
    "street_base",
    "unit",
    "locality"
]

for _, group in (
    address_suffix_audit
    .dropna(
        subset=[
            "street_base",
            "street_suffix",
            "locality"
        ]
    )
    .groupby(
        group_columns,
        dropna=False,
        sort=False
    )
):

    if len(group) < 2:
        continue

    suffixes = set(
        group["street_suffix"]
    )

    nonblank_suffixes = {
        suffix
        for suffix in suffixes
        if suffix
    }

    # Require:
    #
    #   one suffix-free version
    #   +
    #   exactly one known suffix
    #
    # Example:
    #
    #   214 MARSHALL
    #   214 MARSHALL ST
    #
    if (
        "" not in suffixes
        or len(nonblank_suffixes) != 1
    ):
        continue

    address_suffix_candidates.append({
        "entity_type":
            group["entity_type"].iloc[0],

        "canonical_name":
            group["canonical_name"].iloc[0],

        "street_base":
            group["street_base"].iloc[0],

        "locality":
            group["locality"].iloc[0],

        "unit":
            group["unit"].iloc[0],

        "known_suffix":
            next(iter(nonblank_suffixes)),

        "entity_ids":
            tuple(
                group["entity_id"]
            ),

        "addresses":
            tuple(
                group["canonical_address"]
            ),

        "entity_count":
            len(group)
    })


address_suffix_candidates = pd.DataFrame(
    address_suffix_candidates
)

In [66]:
print(
    "Candidate missing-street-suffix groups:",
    len(address_suffix_candidates)
)

address_suffix_candidates

Candidate missing-street-suffix groups: 0


""


In [69]:
entities["occurrence_count"].sum()

np.int64(12058)

In [68]:
entities[
    entities["canonical_name"]
    .str.contains(
        "WIDGER JAMES LEROY",
        regex=False,
        na=False
    )
]

,entity_id,canonical_name,entity_type,canonical_address,organization_type,organization_classification_confidence,occurrence_count
4943,4944,WIDGER JAMES LEROY,Person,"214 MARSHALL ST, ALLEGAN, MI 49010",NaN,NaN,6


In [71]:
entities.head()
entities.info()

<class 'pandas.DataFrame'>
RangeIndex: 5128 entries, 0 to 5127
Data columns (total 7 columns):
 #   Column                                  Non-Null Count  Dtype
---  ------                                  --------------  -----
 0   entity_id                               5128 non-null   int64
 1   canonical_name                          5128 non-null   str  
 2   entity_type                             5128 non-null   str  
 3   canonical_address                       5128 non-null   str  
 4   organization_type                       772 non-null    str  
 5   organization_classification_confidence  772 non-null    str  
 6   occurrence_count                        5128 non-null   int64
dtypes: int64(2), str(5)
memory usage: 280.6 KB


In [72]:
print(
    "Unique source occurrences:",
    entity_occurrences["source_occurrence_id"].nunique()
)

print(
    "Entity occurrences after Joint expansion:",
    len(entity_occurrences)
)

print(
    "Unique occurrence IDs:",
    entity_occurrences["occurrence_id"].nunique()
)

print(
    "Sum of final entity occurrence counts:",
    entities["occurrence_count"].sum()
)

Unique source occurrences: 10270
Entity occurrences after Joint expansion: 12058
Unique occurrence IDs: 12058
Sum of final entity occurrence counts: 12058


In [73]:
print(
    "Expanded Joint-derived occurrences:",
    entity_occurrences["joint_expanded"].sum()
)

Expanded Joint-derived occurrences: 3576


In [74]:
print(
    "Original source rows successfully decomposed:",
    entity_occurrences.loc[
        entity_occurrences["joint_expanded"],
        "source_occurrence_id"
    ].nunique()
)

Original source rows successfully decomposed: 1788
